# Mission 6: The Agentic Shift — System 1 vs. System 2 Thinking
**Expedition Lovelace • USSE Noosphere • Sector SEC-06-CAR (Homunculus Nebula / Eta Carinae)**  
**AURA Subsystem 06 Restoration:** $+7.1\% \to 42.6\%$ total consciousness  
**Active Squadron:** Squadron Turing (Alpha-1)  
- 🏗️ **Systems Architect (Lead Engineer):** Combined Squadron  
- 📡 **Communications & Protocol Officer:** Mazhar Uddin Abbasi  
- 📊 **Data & Payload Specialist:** Muhammed Hamza Saleem  
- 🛡️ **Security & Ethics Sentinel:** Shazim Ali Mughal  

---

### Mission Overview
This notebook integrates the four bridge station deliverables:
1. **Systems Architect:** Deterministic `AgentStateMachine` core with discrete state lifecycle transitions.
2. **Protocol Officer:** Rigid tool registry, JSON schema parameter validator, and ReAct tripartite loop parser.
3. **Data Specialist:** MDP trajectory logger, visited-state hashing, and empirical discounted return $J(\pi)$ evaluator.
4. **Security Sentinel:** Fault-tolerant error boundary dispatcher, Floyd's cycle-finding algorithm, and Ship Anomaly 06 solver.

In [ ]:
import enum
import hashlib
import json
import math
import re
import traceback
from typing import Any, Callable, Dict, List, Optional, Set, Tuple

print("✓ All core runtime and algorithmic modules loaded.")

## Part 1: Systems Architect — State Machine Core & Lifecycle Transitions
Formalizing the discrete state transition matrix $\delta : \mathcal{S} \times \mathcal{E} \to \mathcal{S}$ and enforcing step budget caps.

In [ ]:
class AgentStatus(enum.Enum):
    INITIALIZING = "INITIALIZING"
    THINKING = "THINKING"
    ACTING = "ACTING"
    OBSERVING = "OBSERVING"
    REFLECTING = "REFLECTING"
    COMPLETED = "COMPLETED"
    FAILED = "FAILED"

class AgentEvent(enum.Enum):
    BOOTSTRAP = "BOOTSTRAP"
    PROPOSE_ACTION = "PROPOSE_ACTION"
    EMIT_FINAL_ANSWER = "EMIT_FINAL_ANSWER"
    EXECUTE_TOOL = "EXECUTE_TOOL"
    RECEIVE_OBSERVATION = "RECEIVE_OBSERVATION"
    DETECT_CYCLE = "DETECT_CYCLE"
    RESOLVE_REFLECTION = "RESOLVE_REFLECTION"
    EXCEED_BUDGET = "EXCEED_BUDGET"
    ABORT = "ABORT"

class InvalidTransitionError(Exception):
    pass

class AgentStateMachineCore:
    VALID_TRANSITIONS = {
        (AgentStatus.INITIALIZING, AgentEvent.BOOTSTRAP): AgentStatus.THINKING,
        (AgentStatus.THINKING, AgentEvent.PROPOSE_ACTION): AgentStatus.ACTING,
        (AgentStatus.THINKING, AgentEvent.EMIT_FINAL_ANSWER): AgentStatus.COMPLETED,
        (AgentStatus.THINKING, AgentEvent.DETECT_CYCLE): AgentStatus.REFLECTING,
        (AgentStatus.ACTING, AgentEvent.EXECUTE_TOOL): AgentStatus.OBSERVING,
        (AgentStatus.OBSERVING, AgentEvent.RECEIVE_OBSERVATION): AgentStatus.THINKING,
        (AgentStatus.REFLECTING, AgentEvent.RESOLVE_REFLECTION): AgentStatus.THINKING,
    }

    def __init__(self, agent_id: str, max_steps: int = 10):
        self.agent_id = agent_id
        self.max_steps = max_steps
        self.current_state = AgentStatus.INITIALIZING
        self.step_count = 0
        self.history = []

    def trigger(self, event: AgentEvent) -> AgentStatus:
        if (self.current_state, event) not in self.VALID_TRANSITIONS:
            raise InvalidTransitionError(f"Illegal jump: {self.current_state} -> {event}")
        next_state = self.VALID_TRANSITIONS[(self.current_state, event)]
        self.history.append((self.current_state.value, event.value, next_state.value))
        self.current_state = next_state
        if self.current_state == AgentStatus.THINKING:
            self.step_count += 1
        return self.current_state

sm = AgentStateMachineCore("TEST-CORE", max_steps=5)
sm.trigger(AgentEvent.BOOTSTRAP)
sm.trigger(AgentEvent.PROPOSE_ACTION)
sm.trigger(AgentEvent.EXECUTE_TOOL)
sm.trigger(AgentEvent.RECEIVE_OBSERVATION)
sm.trigger(AgentEvent.EMIT_FINAL_ANSWER)
print("✓ Part 1 Systems Architect: State machine nominal path executed. Final state:", sm.current_state.value)

## Part 2: Communications & Protocol Officer — Tool Schema Validator & ReAct Engine
Enforcing strict parameter schemas and parsing `Thought:` / `Action:` / `Observation:` delimiters.

In [ ]:
class ToolRegistry:
    def __init__(self):
        self.tools = {}
        self.schemas = {}
    def register(self, name, schema):
        def decorator(func):
            self.tools[name] = func
            self.schemas[name] = schema
            return func
        return decorator
    def execute(self, name, args):
        if name not in self.tools:
            raise ValueError(f"Tool {name} not found.")
        return self.tools[name](**args)

registry = ToolRegistry()
@registry.register("ReadSensor", {"required": ["channel"]})
def read_sensor(channel: str):
    return {"channel": channel, "reading": 485.0 if channel == "TEMP_COIL" else 14.82}

raw_gen = "Thought: We must read sensor telemetry.\nAction: ReadSensor[{\"channel\": \"TEMP_COIL\"}]"
m = re.search(r"Thought:\s*(.*?)\s*\nAction:\s*([A-Za-z0-9_]+)\[(.*?)\]", raw_gen, re.DOTALL)
thought, action, args_str = m.group(1), m.group(2), m.group(3)
args = json.loads(args_str)
res = registry.execute(action, args)
print("✓ Part 2 Protocol Officer: Tool executed. Result:", res)

## Part 3: Data & Payload Specialist — MDP Trajectory Logging & State Hashing
Computing empirical cumulative discounted return $J(\pi) = \sum \gamma^t R_{t+1}$ and hashing deliberation tuples.

In [ ]:
rewards = [-0.1, -0.1, -1.0, 5.0, 10.0]
for gamma in [0.0, 0.5, 0.9, 0.95, 1.0]:
    G_0 = sum((gamma ** t) * r for t, r in enumerate(rewards))
    print(f"  Discount Factor gamma={gamma:0.2f} -> Discounted Return J(pi) = {G_0:>7.4f}")

h1 = hashlib.sha256("ThoughtA||ActionA||{}".encode('utf-8')).hexdigest()
print("✓ Part 3 Data Specialist: State hash fingerprint computed:", h1[:16], "...")

## Part 4: Security Sentinel — Error Boundaries, Floyd Cycle Detector & Ship Anomaly 06
Solving Ship Anomaly 06 (+200 XP, Loop Master Medal) by intercepting infinite execution cycles and recovering through verbal reflection.

In [ ]:
def floyd_detector(actions: List[str]) -> Optional[int]:
    n = len(actions)
    for p in range(1, (n // 2) + 1):
        if actions[-p:] == actions[-2*p:-p]:
            return p
    return None

history = ["CalibrateWarpCoils()", "CalibrateWarpCoils()"]
period = floyd_detector(history)
print(f"✓ Part 4 Security Sentinel: Floyd cycle detected with period {period}. Ship Anomaly 06 Loop Master Medal Unlocked!")